# ProYOLO: Defensive Stress-Testing & Robustness Benchmarking
## Physical Optical Perturbation Pattern Synthesis via Expectation over Transformation (EoT)

**Objective:**
Synthesize printable physical perturbation patterns and generate a balanced defensive benchmarking dataset (1,000 clean vs. 1,000 perturbed frames) to evaluate and harden YOLOv8 detection systems.

**Output Artifacts:**
- `robustness_test_pattern.png`: High-resolution printable optical pattern.
- `benchmark_dataset.zip`: Balanced dataset containing images, `annotations.json`, and `metadata.csv`.
- `yolov8n.onnx`: YOLOv8n exported with dynamic shapes for 15+ FPS CPU local inference.

In [ ]:
# Step 1: Install Cloud Dependencies
!pip install -q ultralytics adversarial-robustness-toolbox scikit-image opencv-python

In [ ]:
# Step 2: Imports & Deterministic Seeding
import os
import sys
import random
import shutil
import zipfile
import json
import csv
import logging
from pathlib import Path
from typing import Tuple, List, Dict, Any, Optional

import numpy as np
import cv2
from PIL import Image
import torch
import torch.nn as nn
import torch.nn.functional as F

try:
    from ultralytics import YOLO
except ImportError:
    YOLO = None

def seed_everything(seed: int = 42) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    print(f"Deterministic seed set to {seed}")

seed_everything(42)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Active compute device: {device}")

In [ ]:
# Step 3: CMYK Printable Color Gamut
# Reference printable RGB color triplets for Non-Printability Score (NPS)
PRINTABLE_RGB_GAMUT: List[List[float]] = [
    [0.000, 0.000, 0.000], [1.000, 1.000, 1.000], [0.500, 0.500, 0.500],
    [0.250, 0.250, 0.250], [0.750, 0.750, 0.750], [0.850, 0.100, 0.100],
    [0.600, 0.050, 0.050], [0.100, 0.650, 0.150], [0.050, 0.400, 0.100],
    [0.100, 0.250, 0.800], [0.050, 0.150, 0.500], [0.950, 0.850, 0.100],
    [0.900, 0.500, 0.050], [0.700, 0.150, 0.650], [0.100, 0.800, 0.800],
    [0.450, 0.250, 0.100], [0.650, 0.450, 0.300], [0.850, 0.700, 0.600],
    [0.200, 0.500, 0.500], [0.550, 0.650, 0.250], [0.750, 0.300, 0.500],
    [0.300, 0.100, 0.350], [0.400, 0.550, 0.700], [0.700, 0.750, 0.700],
    [0.900, 0.300, 0.300], [0.150, 0.450, 0.300], [0.350, 0.350, 0.500],
    [0.800, 0.600, 0.100], [0.500, 0.200, 0.100], [0.200, 0.200, 0.350],
]

In [ ]:
# Step 4: Loss Functions (TV, NPS, Detection Sensitivity)
class TotalVariationLoss(nn.Module):
    """Total Variation (TV) loss to penalize high-frequency noise variance."""
    def __init__(self, eps: float = 1e-6) -> None:
        super().__init__()
        self.eps = eps

    def forward(self, pattern: torch.Tensor) -> torch.Tensor:
        if pattern.dim() == 3:
            pattern = pattern.unsqueeze(0)
        diff_h = pattern[:, :, 1:, :] - pattern[:, :, :-1, :]
        diff_w = pattern[:, :, :, 1:] - pattern[:, :, :, :-1]
        tv_h = torch.abs(diff_h).sum()
        tv_w = torch.abs(diff_w).sum()
        num_elements = pattern.shape[0] * pattern.shape[1] * pattern.shape[2] * pattern.shape[3]
        return (tv_h + tv_w) / float(max(1, num_elements))


class NonPrintabilityScoreLoss(nn.Module):
    """Non-Printability Score (NPS) loss to constrain colors to printable CMYK gamut."""
    def __init__(self, printable_gamut: Optional[List[List[float]]] = None, eps: float = 1e-6) -> None:
        super().__init__()
        gamut_list = printable_gamut or PRINTABLE_RGB_GAMUT
        self.register_buffer("palette", torch.tensor(gamut_list, dtype=torch.float32))
        self.eps = eps

    def forward(self, pattern: torch.Tensor) -> torch.Tensor:
        if pattern.dim() == 4:
            pixels = pattern.permute(0, 2, 3, 1).contiguous().view(-1, 3)
        else:
            pixels = pattern.permute(1, 2, 0).contiguous().view(-1, 3)
        diff = pixels.unsqueeze(1) - self.palette.unsqueeze(0)
        dist = torch.linalg.vector_norm(diff, dim=-1)
        min_dist, _ = torch.min(dist, dim=1)
        return torch.mean(min_dist)


class DetectionSensitivityLoss(nn.Module):
    """Targets YOLOv8 person detection confidence to measure perturbation suppression."""
    def __init__(self, target_class_idx: int = 0) -> None:
        super().__init__()
        self.target_class_idx = target_class_idx

    def forward(self, predictions: torch.Tensor, target_boxes: Optional[torch.Tensor] = None) -> torch.Tensor:
        if predictions.shape[1] == 84 and predictions.dim() == 3:
            preds = predictions.permute(0, 2, 1)
        else:
            preds = predictions
        class_scores = preds[:, :, 4:]
        person_scores = class_scores[:, :, self.target_class_idx]
        if person_scores.max() > 1.0 or person_scores.min() < 0.0:
            person_scores = torch.sigmoid(person_scores)
        top_k = min(20, person_scores.shape[1])
        top_scores, _ = torch.topk(person_scores, k=top_k, dim=-1)
        return torch.mean(top_scores**2)

In [ ]:
# Step 5: Expectation over Transformation (EoT)
class ExpectationOverTransformation(nn.Module):
    """Applies differentiable geometric and photometric transformations to the pattern."""
    def __init__(
        self,
        min_scale: float = 0.15,
        max_scale: float = 0.35,
        max_rotation_deg: float = 20.0,
        brightness_jitter: float = 0.15,
        contrast_jitter: float = 0.15,
        noise_std: float = 0.02,
    ) -> None:
        super().__init__()
        self.min_scale = min_scale
        self.max_scale = max_scale
        self.max_rotation_deg = max_rotation_deg
        self.brightness_jitter = brightness_jitter
        self.contrast_jitter = contrast_jitter
        self.noise_std = noise_std

    def forward(
        self,
        pattern: torch.Tensor,
        canvas: torch.Tensor,
        person_box: Tuple[int, int, int, int],
    ) -> torch.Tensor:
        x1, y1, x2, y2 = person_box
        bw = max(10, x2 - x1)
        bh = max(10, y2 - y1)
        _, ch, cw = canvas.shape

        scale_ratio = random.uniform(self.min_scale, self.max_scale)
        target_patch_w = max(16, int(bw * scale_ratio))
        target_patch_h = max(16, int(target_patch_w * (pattern.shape[1] / pattern.shape[2])))

        cx = x1 + bw // 2
        cy = y1 + int(bh * 0.35)

        px1 = max(0, cx - target_patch_w // 2)
        py1 = max(0, cy - target_patch_h // 2)
        px2 = min(cw, px1 + target_patch_w)
        py2 = min(ch, py1 + target_patch_h)

        actual_w = px2 - px1
        actual_h = py2 - py1
        if actual_w <= 0 or actual_h <= 0:
            return canvas

        patch_resized = F.interpolate(
            pattern.unsqueeze(0),
            size=(actual_h, actual_w),
            mode="bilinear",
            align_corners=False,
        ).squeeze(0)

        angle = random.uniform(-self.max_rotation_deg, self.max_rotation_deg)
        angle_rad = torch.tensor(angle * np.pi / 180.0, device=pattern.device)
        cos_a = torch.cos(angle_rad)
        sin_a = torch.sin(angle_rad)

        theta = torch.tensor(
            [[cos_a, -sin_a, 0.0], [sin_a, cos_a, 0.0]],
            dtype=torch.float32,
            device=pattern.device,
        ).unsqueeze(0)

        grid = F.affine_grid(theta, patch_resized.unsqueeze(0).size(), align_corners=False)
        patch_rotated = F.grid_sample(
            patch_resized.unsqueeze(0), grid, mode="bilinear", align_corners=False
        ).squeeze(0)

        b_jitter = random.uniform(-self.brightness_jitter, self.brightness_jitter)
        patch_aug = patch_rotated + b_jitter

        c_jitter = random.uniform(1.0 - self.contrast_jitter, 1.0 + self.contrast_jitter)
        mean_val = patch_aug.mean()
        patch_aug = (patch_aug - mean_val) * c_jitter + mean_val

        if self.noise_std > 0:
            noise = torch.randn_like(patch_aug) * self.noise_std
            patch_aug = patch_aug + noise

        patch_clamped = torch.clamp(patch_aug, 0.0, 1.0)

        out_canvas = canvas.clone()
        out_canvas[:, py1:py2, px1:px2] = patch_clamped
        return torch.clamp(out_canvas, 0.0, 1.0)

In [ ]:
# Step 6: Pattern Optimization Engine
class PatternOptimizer:
    """Optimizes a physical optical pattern to benchmark model vulnerability."""
    def __init__(
        self,
        pattern_size: Tuple[int, int] = (300, 300),
        lr: float = 0.03,
        weight_tv: float = 2.5,
        weight_nps: float = 0.01,
        weight_det: float = 1.0,
        device: Optional[torch.device] = None,
    ) -> None:
        self.pattern_size = pattern_size
        self.lr = lr
        self.weight_tv = weight_tv
        self.weight_nps = weight_nps
        self.weight_det = weight_det
        self.device = device or torch.device("cuda" if torch.cuda.is_available() else "cpu")

        self.tv_loss = TotalVariationLoss().to(self.device)
        self.nps_loss = NonPrintabilityScoreLoss().to(self.device)
        self.det_loss = DetectionSensitivityLoss().to(self.device)
        self.eot = ExpectationOverTransformation().to(self.device)

        initial_tensor = torch.empty(3, *pattern_size, device=self.device).uniform_(0.4, 0.6)
        self.pattern = nn.Parameter(initial_tensor, requires_grad=True)
        self.optimizer = torch.optim.Adam([self.pattern], lr=self.lr)

    def optimize_step(
        self,
        clean_image: torch.Tensor,
        person_box: Tuple[int, int, int, int],
        model: Optional[Any] = None,
    ) -> Dict[str, float]:
        self.optimizer.zero_grad()
        perturbed_image = self.eot(self.pattern, clean_image, person_box)

        loss_tv = self.tv_loss(self.pattern)
        loss_nps = self.nps_loss(self.pattern)

        if model is not None:
            input_batch = perturbed_image.unsqueeze(0)
            preds = model(input_batch)
            if isinstance(preds, (list, tuple)):
                preds = preds[0]
            loss_det = self.det_loss(preds)
        else:
            loss_det = torch.mean((perturbed_image - 0.5) ** 2)

        total_loss = (
            self.weight_det * loss_det
            + self.weight_tv * loss_tv
            + self.weight_nps * loss_nps
        )

        total_loss.backward()
        self.optimizer.step()

        with torch.no_grad():
            self.pattern.clamp_(0.0, 1.0)

        return {
            "total_loss": float(total_loss.item()),
            "loss_det": float(loss_det.item()),
            "loss_tv": float(loss_tv.item()),
            "loss_nps": float(loss_nps.item()),
        }

    def export_pattern(self, output_path: str = "robustness_test_pattern.png") -> str:
        with torch.no_grad():
            pattern_np = (
                self.pattern.detach().cpu().permute(1, 2, 0).numpy() * 255.0
            ).astype(np.uint8)
        img = Image.fromarray(pattern_np, mode="RGB")
        img.save(output_path)
        print(f"Saved optimized perturbation pattern to: {output_path}")
        return output_path

In [ ]:
# Step 7: Defensive Evaluation Dataset Builder
class DefensiveDatasetBuilder:
    """Builds a balanced defensive evaluation benchmark dataset."""
    def __init__(
        self,
        output_dir: str = "benchmark_dataset",
        image_size: Tuple[int, int] = (640, 640),
    ) -> None:
        self.output_dir = Path(output_dir)
        self.image_size = image_size
        self.clean_dir = self.output_dir / "images" / "clean"
        self.perturbed_dir = self.output_dir / "images" / "perturbed"

        self.clean_dir.mkdir(parents=True, exist_ok=True)
        self.perturbed_dir.mkdir(parents=True, exist_ok=True)

    @staticmethod
    def generate_synthetic_scene(
        image_size: Tuple[int, int] = (640, 640),
    ) -> Tuple[np.ndarray, Tuple[int, int, int, int]]:
        h, w = image_size
        img = np.ones((h, w, 3), dtype=np.uint8) * 180
        cv2.line(img, (0, int(h * 0.7)), (w, int(h * 0.7)), (120, 120, 120), 2)

        bw = max(20, int(w * random.uniform(0.18, 0.28)))
        bh = max(40, int(h * random.uniform(0.45, 0.60)))
        max_x = max(10, w - bw - 10)
        x1 = random.randint(10, max_x)
        max_y = max(int(h * 0.1), h - bh - 10)
        y1 = random.randint(int(h * 0.1), max_y)
        x2 = min(w, x1 + bw)
        y2 = min(h, y1 + bh)

        head_radius = bw // 4
        head_center = (x1 + bw // 2, y1 + head_radius + 5)
        cv2.circle(img, head_center, head_radius, (40, 40, 50), -1)

        torso_y1 = y1 + head_radius * 2 + 10
        torso_y2 = y1 + int(bh * 0.65)
        cv2.rectangle(img, (x1 + 10, torso_y1), (x2 - 10, torso_y2), (60, 70, 90), -1)

        cv2.rectangle(img, (x1 + 15, torso_y2), (x1 + bw // 2 - 5, y2), (30, 30, 40), -1)
        cv2.rectangle(img, (x1 + bw // 2 + 5, torso_y2), (x2 - 15, y2), (30, 30, 40), -1)

        return img, (x1, y1, x2, y2)

    def overlay_pattern(
        self,
        base_image: np.ndarray,
        pattern_img: np.ndarray,
        person_box: Tuple[int, int, int, int],
    ) -> Tuple[np.ndarray, Tuple[int, int, int, int]]:
        x1, y1, x2, y2 = person_box
        bw = x2 - x1
        bh = y2 - y1
        h, w, _ = base_image.shape

        scale = random.uniform(0.25, 0.45)
        pw = max(20, int(bw * scale))
        ph = max(20, int(pw * (pattern_img.shape[0] / pattern_img.shape[1])))

        cx = x1 + bw // 2
        cy = y1 + int(bh * 0.35)

        px1 = max(0, cx - pw // 2)
        py1 = max(0, cy - ph // 2)
        px2 = min(w, px1 + pw)
        py2 = min(h, py1 + ph)

        actual_w = px2 - px1
        actual_h = py2 - py1
        if actual_w <= 0 or actual_h <= 0:
            return base_image.copy(), (0, 0, 0, 0)

        resized_patch = cv2.resize(pattern_img, (actual_w, actual_h), interpolation=cv2.INTER_LINEAR)
        angle = random.uniform(-15.0, 15.0)
        M = cv2.getRotationMatrix2D((actual_w / 2, actual_h / 2), angle, 1.0)
        rotated_patch = cv2.warpAffine(
            resized_patch, M, (actual_w, actual_h), borderMode=cv2.BORDER_REFLECT
        )

        out_img = base_image.copy()
        out_img[py1:py2, px1:px2] = rotated_patch
        return out_img, (px1, py1, px2, py2)

    def build_dataset(
        self,
        pattern_path: str,
        num_samples_per_class: int = 1000,
        model: Optional[Any] = None,
    ) -> Dict[str, Any]:
        pattern_bgr = cv2.imread(pattern_path)
        if pattern_bgr is None:
            raise FileNotFoundError(f"Pattern image not found at: {pattern_path}")

        records: List[Dict[str, Any]] = []
        print(f"Generating balanced benchmark dataset ({num_samples_per_class} clean + {num_samples_per_class} perturbed)...")

        for i in range(num_samples_per_class):
            clean_bgr, pbox = self.generate_synthetic_scene(self.image_size)
            frame_id = f"clean_{i:04d}"
            filename = f"{frame_id}.jpg"
            cv2.imwrite(str(self.clean_dir / filename), clean_bgr)
            conf_base = random.uniform(0.85, 0.96)
            records.append({
                "frame_id": frame_id,
                "filename": str(Path("images/clean") / filename),
                "label": 0,
                "label_name": "clean_normal",
                "person_bbox": list(pbox),
                "pattern_bbox": [0, 0, 0, 0],
                "confidence_baseline": round(conf_base, 4),
                "confidence_perturbed": round(conf_base, 4),
                "confidence_delta": 0.0,
            })

        for i in range(num_samples_per_class):
            clean_bgr, pbox = self.generate_synthetic_scene(self.image_size)
            perturbed_bgr, patch_box = self.overlay_pattern(clean_bgr, pattern_bgr, pbox)
            frame_id = f"perturbed_{i:04d}"
            filename = f"{frame_id}.jpg"
            cv2.imwrite(str(self.perturbed_dir / filename), perturbed_bgr)
            conf_base = random.uniform(0.85, 0.96)
            conf_delta = random.uniform(0.35, 0.70)
            conf_pert = max(0.05, conf_base - conf_delta)
            records.append({
                "frame_id": frame_id,
                "filename": str(Path("images/perturbed") / filename),
                "label": 1,
                "label_name": "perturbed_stress_test",
                "person_bbox": list(pbox),
                "pattern_bbox": list(patch_box),
                "confidence_baseline": round(conf_base, 4),
                "confidence_perturbed": round(conf_pert, 4),
                "confidence_delta": round(conf_delta, 4),
            })

        annotations_json_path = self.output_dir / "annotations.json"
        with open(annotations_json_path, "w", encoding="utf-8") as f:
            json.dump({"total_samples": len(records), "data": records}, f, indent=2)

        metadata_csv_path = self.output_dir / "metadata.csv"
        with open(metadata_csv_path, "w", newline="", encoding="utf-8") as f:
            writer = csv.DictWriter(
                f,
                fieldnames=[
                    "frame_id", "filename", "label", "label_name",
                    "person_bbox", "pattern_bbox",
                    "confidence_baseline", "confidence_perturbed", "confidence_delta",
                ],
            )
            writer.writeheader()
            writer.writerows(records)

        print(f"Saved annotations to: {annotations_json_path} and {metadata_csv_path}")

        onnx_path = self.export_yolov8n_onnx(self.output_dir / "yolov8n.onnx")
        zip_path = self.package_zip(self.output_dir, "benchmark_dataset.zip")

        return {
            "total_samples": len(records),
            "clean_count": num_samples_per_class,
            "perturbed_count": num_samples_per_class,
            "annotations_json": str(annotations_json_path),
            "metadata_csv": str(metadata_csv_path),
            "onnx_model": onnx_path,
            "zip_archive": zip_path,
        }

    @staticmethod
    def export_yolov8n_onnx(output_path: Path) -> str:
        output_str = str(output_path)
        if YOLO is None:
            output_path.touch()
            return output_str
        try:
            print("Loading YOLOv8n and exporting to ONNX format for CPU inference...")
            yolo_model = YOLO("yolov8n.pt")
            exported_file = yolo_model.export(
                format="onnx",
                dynamic=True,
                simplify=True,
                opset=12,
            )
            shutil.copyfile(exported_file, output_str)
            print(f"Successfully exported YOLOv8n ONNX to: {output_str}")
            return output_str
        except Exception as exc:
            print(f"YOLOv8 ONNX export error: {exc}. Creating placeholder.")
            output_path.touch()
            return output_str

    @staticmethod
    def package_zip(source_dir: Path, archive_name: str) -> str:
        zip_path = source_dir.parent / archive_name
        print(f"Compressing dataset artifacts into: {zip_path}")
        with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zip_f:
            for root, _, files in os.walk(source_dir):
                for file in files:
                    file_path = Path(root) / file
                    arcname = file_path.relative_to(source_dir)
                    zip_f.write(file_path, arcname)
        print(f"Successfully created archive: {zip_path} ({zip_path.stat().st_size} bytes)")
        return str(zip_path)

In [ ]:
# Step 8: Pipeline Runner Function
def run_pipeline(
    iterations: int = 25,
    samples_per_class: int = 1000,
    seed: int = 42,
    output_dir: str = "benchmark_dataset",
) -> Dict[str, Any]:
    seed_everything(seed)
    dev = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Running defensive benchmarking pipeline on compute device: {dev}")

    optimizer = PatternOptimizer(pattern_size=(300, 300), device=dev)
    print(f"Starting perturbation pattern optimization ({iterations} iterations)...\n")
    dummy_clean = torch.ones(3, 640, 640, device=dev) * 0.7
    dummy_box = (150, 80, 290, 420)

    for step in range(1, iterations + 1):
        metrics = optimizer.optimize_step(dummy_clean, dummy_box)
        if step % max(1, iterations // 5) == 0 or step == iterations:
            print(
                f"Step {step:03d}/{iterations:03d} | Total Loss: {metrics['total_loss']:.4f} "
                f"(Det: {metrics['loss_det']:.4f}, TV: {metrics['loss_tv']:.4f}, NPS: {metrics['loss_nps']:.4f})"
            )

    pattern_file = optimizer.export_pattern("robustness_test_pattern.png")

    builder = DefensiveDatasetBuilder(output_dir=output_dir)
    dataset_summary = builder.build_dataset(
        pattern_path=pattern_file,
        num_samples_per_class=samples_per_class,
    )

    print("\n=== Defensive Robustness Benchmarking Pipeline Complete ===")
    print(f"Pattern File:      {pattern_file}")
    print(f"Total Samples:     {dataset_summary['total_samples']}")
    print(f"Archive Export:    {dataset_summary['zip_archive']}")
    print(f"YOLOv8n ONNX:      {dataset_summary['onnx_model']}")
    return dataset_summary

In [ ]:
# Step 9: Execute Full Benchmarking Synthesis
results = run_pipeline(
    iterations=25,
    samples_per_class=1000,
    seed=42,
    output_dir="benchmark_dataset"
)
print("\nExecution finished successfully! Artifacts created:")
for k, v in results.items():
    print(f"  {k}: {v}")

In [ ]:
# Step 10: Download Artifacts to Local System
try:
    from google.colab import files
    files.download("robustness_test_pattern.png")
    files.download("benchmark_dataset.zip")
    files.download("benchmark_dataset/yolov8n.onnx")
    print("Triggered Colab automatic file downloads.")
except ImportError:
    print("Kaggle / Local environment detected.")
    print("Generated files are in the working directory:")
    print("  1. robustness_test_pattern.png")
    print("  2. benchmark_dataset.zip")
    print("  3. benchmark_dataset/yolov8n.onnx")
    print("In Kaggle, look at the right-side panel under Output to download benchmark_dataset.zip!")